# Biomarker Analysis Pipeline

Runs all cohort combinations:
- **Cohorts**: `all_ICI` (any-line ICI vs never-ICI), `first_line` (first-line ICI vs never-ICI)

Each run produces 3 sensitivity specs internally: stabilized ATE, stabilized ATT, and unweighted (noIPTW).

### Stages
1. **Propensity scores** — `ICI_LRs_all_ICI.py`, `ICI_LRs_first_line.py`
2. **IPTW datasets** — `generate_IPTW_df.py --cohort {cohort}`
3. **Cox models** — `run_IPTW_analysis.py --cohort {cohort}`

In [ ]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
COHORTS = ['all_ICI', 'first_line']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Propensity Score Generation

In [ ]:
run_and_stream('all_ICI propensity', [sys.executable, 'ICI_LRs_all_ICI.py'])
run_and_stream('first_line propensity', [sys.executable, 'ICI_LRs_first_line.py'])

## Stage 2: IPTW Dataset Generation

In [ ]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} IPTW dataset',
                   [sys.executable, 'generate_IPTW_df.py', '--cohort', cohort])

## Stage 3: IPTW Cox Model Analysis

In [ ]:
for cohort in COHORTS:
    run_and_stream(f'{cohort} Cox models',
                   [sys.executable, 'run_IPTW_analysis.py', '--cohort', cohort])